# 01 — ข้อมูล, Split และ EDA: Seoul Bike Demand

| ขั้น | เนื้อหา | ใช้ข้อมูล |
|---|---|---|
| 2 | โหลดและตรวจโครงสร้างข้อมูล | ทั้งชุด (ตรวจโครงสร้างเท่านั้น ไม่ดูความสัมพันธ์กับ target) |
| 3 | แบ่ง train/test **ตามวัน** → `data/splits/` | ทั้งชุด |
| 4 | EDA + การตัดสินใจที่จะใช้ใน `02_modeling` | **train เท่านั้น** |

> ⚠️ **ทุกการตัดสินใจ (ตัดแถว, outlier, threshold, เลือก feature) ต้องมาจาก train เท่านั้น** — test ปิดไว้จนถึงขั้นประเมินครั้งเดียวใน `02_modeling`

### Setup
**โค้ดทำอะไร:** import ไลบรารี, หา root ของ repo (รันได้ทั้งจากโฟลเดอร์ `notebooks/` และจาก root) แล้ว import `config.py` เพื่อใช้ค่าคงที่ชุดเดียวกับทุก notebook
ฟังก์ชัน `savefig()` บันทึกกราฟลง `figures/` เพื่อใช้ในรายงาน

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")


def savefig(name):
    plt.savefig(config.FIGURES_DIR / name, dpi=120, bbox_inches="tight")


print("TASK =", config.TASK, "| RANDOM_STATE =", config.RANDOM_STATE)

TASK = classification | RANDOM_STATE = 42


---
## ขั้น 2 — โหลดและตรวจข้อมูล

### 2.1 ตรวจ encoding และรูปแบบวันที่ก่อนโหลดจริง
**โค้ดทำอะไร:**
1. อ่านบรรทัดแรกของไฟล์เป็น **bytes ดิบ** เพื่อดูว่าสัญลักษณ์ `°` ถูกเก็บเป็น byte อะไร
2. ลองอ่านหัวตาราง (`nrows=0`) ด้วย 3 encoding แล้วดูว่าแบบไหนอ่านชื่อคอลัมน์ `Temperature(°C)` ได้ถูก
3. แยกคอลัมน์วันที่ดิบด้วย `/` แล้วดูช่วงค่าของแต่ละส่วน — ส่วนที่มีค่าเกิน 12 ต้องเป็น "วัน" ไม่ใช่ "เดือน"

**ทำไมต้องทำ:** ถ้าปล่อยให้ pandas เดา อาจอ่าน `°` เพี้ยน หรือสลับวัน/เดือน (01/12 = 1 ธ.ค. หรือ 12 ม.ค.?) โดยไม่มี error เตือนเลย

In [2]:
header_bytes = config.RAW_CSV.read_bytes().split(b"\r\n")[0]
print(header_bytes[:60], "...")

for enc in ["utf-8", "cp1252", "latin-1"]:
    try:
        cols = pd.read_csv(config.RAW_CSV, encoding=enc, nrows=0).columns
        print(f"{enc:<8} -> {cols[3]!r}")
    except UnicodeDecodeError as e:
        print(f"{enc:<8} -> UnicodeDecodeError: {e.reason} (byte {e.object[e.start:e.start + 1]!r})")

date_raw = pd.read_csv(config.RAW_CSV, encoding=config.RAW_ENCODING, usecols=["Date"])["Date"]
parts = date_raw.str.split("/", expand=True).astype(int)
print("\nตัวอย่างวันที่ดิบ:", date_raw.iloc[0], "...", date_raw.iloc[-1])
for i, name in enumerate(["ส่วนที่ 1", "ส่วนที่ 2", "ส่วนที่ 3"]):
    print(f"{name}: min={parts[i].min():>4}  max={parts[i].max():>4}")

b'Date,Rented Bike Count,Hour,Temperature(\xb0C),Humidity(%),Wind' ...
utf-8    -> UnicodeDecodeError: invalid start byte (byte b'\xb0')
cp1252   -> 'Temperature(°C)'
latin-1  -> 'Temperature(°C)'

ตัวอย่างวันที่ดิบ: 01/12/2017 ... 30/11/2018
ส่วนที่ 1: min=   1  max=  31
ส่วนที่ 2: min=   1  max=  12
ส่วนที่ 3: min=2017  max=2018


**ผลลัพธ์:**
- byte ของ `°` คือ `\xb0` → เป็น encoding แบบ single-byte ของ Windows ไม่ใช่ UTF-8 (UTF-8 จะเก็บ `°` เป็น 2 bytes `\xc2\xb0`) จึงทำให้ `utf-8` อ่านไม่ได้เลย (`UnicodeDecodeError`)
- `cp1252` และ `latin-1` อ่านได้ `Temperature(°C)` ถูกต้องทั้งคู่ (สำหรับ byte นี้สองแบบให้ผลเหมือนกัน) → เลือก **`cp1252`** เพราะไฟล์มาจาก Excel บน Windows · บันทึกไว้ที่ `config.RAW_ENCODING`
- วันที่: ส่วนที่ 1 มีค่าถึง 31 และส่วนที่ 2 ไม่เกิน 12 → รูปแบบคือ **`dd/mm/yyyy`** (`config.DATE_FORMAT = "%d/%m/%Y"`) และแถวแรกคือ 1 ธ.ค. 2017 ไม่ใช่ 12 ม.ค.

### 2.2 โหลดข้อมูล + เปลี่ยนชื่อคอลัมน์
**โค้ดทำอะไร:**
- อ่าน CSV ด้วย encoding ที่ตรวจแล้ว
- `assert` ว่าชื่อคอลัมน์ดิบตรงกับ `config.COLUMN_MAP` ทุกตัว (ถ้าไฟล์เปลี่ยน notebook จะหยุดทันที แทนที่จะรันต่อแบบผิดๆ)
- เปลี่ยนชื่อเป็น snake_case ไม่มีหน่วย/อักขระพิเศษ (เช่น `Temperature(°C)` → `temp_c`) ลดบั๊กจาก `°`, วงเล็บ, ช่องว่าง
- แปลง `date` เป็น datetime โดย **ระบุ `format=` ชัดเจน**

In [3]:
df = pd.read_csv(config.RAW_CSV, encoding=config.RAW_ENCODING)
assert list(df.columns) == list(config.COLUMN_MAP), "ชื่อคอลัมน์ไม่ตรงกับ config.COLUMN_MAP"
df = df.rename(columns=config.COLUMN_MAP)
df["date"] = pd.to_datetime(df["date"], format=config.DATE_FORMAT)

print("shape:", df.shape)
df.head()

shape: (8760, 14)


,date,rented_bike_count,hour,temp_c,humidity_pct,wind_speed_ms,visibility_10m,dew_point_c,solar_radiation_mj,rainfall_mm,snowfall_cm,seasons,holiday,functioning_day
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


**ผลลัพธ์:** ได้ **8,760 แถว × 14 คอลัมน์** ตรงกับที่คาด (365 วัน × 24 ชั่วโมง) แถวแรกคือ 1 ธ.ค. 2017 เวลา 0:00 ยอดเช่า 254 คัน อุณหภูมิ −5.2 °C

**โค้ดทำอะไร:** ดูชนิดข้อมูล (dtype) และจำนวนค่าที่ไม่ว่างของทุกคอลัมน์ด้วย `df.info()`

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                8760 non-null   datetime64[ns]
 1   rented_bike_count   8760 non-null   int64         
 2   hour                8760 non-null   int64         
 3   temp_c              8760 non-null   float64       
 4   humidity_pct        8760 non-null   int64         
 5   wind_speed_ms       8760 non-null   float64       
 6   visibility_10m      8760 non-null   int64         
 7   dew_point_c         8760 non-null   float64       
 8   solar_radiation_mj  8760 non-null   float64       
 9   rainfall_mm         8760 non-null   float64       
 10  snowfall_cm         8760 non-null   float64       
 11  seasons             8760 non-null   object        
 12  holiday             8760 non-null   object        
 13  functioning_day     8760 non-null   object      

**ผลลัพธ์:**
- `date` เป็น `datetime64` แล้ว (แปลงสำเร็จ)
- ตัวเลข: `rented_bike_count`, `hour`, `humidity_pct`, `visibility_10m` เป็น int · สภาพอากาศที่เหลือเป็น float
- `seasons`, `holiday`, `functioning_day` เป็นข้อความ (`object`) → ต้อง encode ก่อนเข้าโมเดล (ขั้น 5–6)
- ทุกคอลัมน์มี **8,760 non-null** = ไม่มี missing value เลย

### 2.3 ตรวจความครบถ้วนของข้อมูล
**โค้ดทำอะไร:** รวมผลตรวจหลายข้อไว้ในตารางเดียว แล้ว `assert` ข้อที่ต้องจริงเสมอ
- missing, แถวซ้ำทั้งแถว และแถวซ้ำที่ `(date, hour)` เดียวกัน (ชั่วโมงเดียวกันไม่ควรมี 2 ค่า)
- ช่วงวันที่, จำนวนวัน, จำนวนแถวต่อวัน และทุกวันมีชั่วโมง 0–23 ครบไหม

In [5]:
rows_per_day = df.groupby("date").size()
hours_complete = df.groupby("date")["hour"].apply(lambda h: sorted(h) == list(range(24))).all()

checks = pd.Series({
    "จำนวนแถว": len(df),
    "missing (รวมทุกคอลัมน์)": int(df.isna().sum().sum()),
    "แถวซ้ำทั้งแถว": int(df.duplicated().sum()),
    "แถวซ้ำ (date, hour)": int(df.duplicated(["date", "hour"]).sum()),
    "วันแรก": df["date"].min().date(),
    "วันสุดท้าย": df["date"].max().date(),
    "จำนวนวัน": df["date"].nunique(),
    "แถวต่อวัน (min–max)": f"{rows_per_day.min()}–{rows_per_day.max()}",
    "ทุกวันมีชั่วโมง 0–23 ครบ": hours_complete,
}, dtype=object)

assert len(df) == 8760 and df["date"].nunique() == 365 and hours_complete
assert df.isna().sum().sum() == 0 and df.duplicated(["date", "hour"]).sum() == 0
checks.to_frame("ค่า")

,ค่า
จำนวนแถว,8760
missing (รวมทุกคอลัมน์),0
แถวซ้ำทั้งแถว,0
"แถวซ้ำ (date, hour)",0
วันแรก,2017-12-01
วันสุดท้าย,2018-11-30
จำนวนวัน,365
แถวต่อวัน (min–max),24–24
ทุกวันมีชั่วโมง 0–23 ครบ,True


**ผลลัพธ์:** ข้อมูลสะอาดเชิงโครงสร้างครบทุกข้อ
- ไม่มี missing, ไม่มีแถวซ้ำ (ยืนยันสิ่งที่ UCI บอกไว้ด้วยตัวเอง)
- ครอบคลุม **1 ธ.ค. 2017 – 30 พ.ย. 2018 = 365 วัน** ทุกวันมี 24 แถว ชั่วโมง 0–23 ครบ
- → ใช้ "วันที่" เป็นหน่วยของการ split ได้อย่างปลอดภัย (ทุกกลุ่มมีขนาดเท่ากัน 24 แถว)

### 2.4 ค่าที่เป็นไปได้ของคอลัมน์หมวดหมู่
**โค้ดทำอะไร:** นับจำนวนแถวของแต่ละค่าในคอลัมน์ข้อความ เพื่อตรวจการสะกด/ค่าแปลกปลอม (เช่น `"Yes "` ที่มีช่องว่าง) ก่อนนำไป encode

In [6]:
for c in ["seasons", "holiday", "functioning_day"]:
    print(f"{c:<16}", df[c].value_counts().to_dict())

seasons          {'Spring': 2208, 'Summer': 2208, 'Autumn': 2184, 'Winter': 2160}
holiday          {'No Holiday': 8328, 'Holiday': 432}
functioning_day  {'Yes': 8465, 'No': 295}


**ผลลัพธ์:**
- `seasons` มี 4 ค่า (Spring/Summer/Autumn/Winter) จำนวนแถวใกล้กัน (2,160–2,208) เพราะแต่ละฤดูยาว ~90 วัน
- `holiday` มี 2 ค่า: `Holiday` 432 แถว = 18 วันหยุดนักขัตฤกษ์ · `No Holiday` 8,328 แถว
- `functioning_day` มี 2 ค่า: `No` **295 แถว** คือชั่วโมงที่ระบบปิดให้บริการ — จะวิเคราะห์ในขั้น 4 (บน train)
- ไม่มีค่าสะกดผิด/ช่องว่างเกิน → map เป็นตัวเลขได้ตรงๆ

**สรุปขั้น 2 (จดลง `data/README.md` แล้ว):** 8,760 แถว × 14 คอลัมน์, encoding `cp1252`, วันที่ `dd/mm/yyyy`, 365 วัน × 24 ชม., ไม่มี missing/ซ้ำ

---
## ขั้น 3 — Split ตามวัน ⚠️ ขั้นที่สำคัญที่สุด

**ทำไมห้ามสุ่มทีละแถว** ชั่วโมง 14:00 กับ 15:00 ของวันเดียวกันมีอากาศแทบเหมือนกันและยอดเช่าใกล้กัน
ถ้า 14:00 อยู่ใน train และ 15:00 อยู่ใน test โมเดลแค่ "จำ" วันนั้นได้ → คะแนน test สูงเกินจริง = **leakage แบบซ่อน**
วิธีแก้คือให้ **ทั้งวัน (24 ชม.) อยู่ฝั่งเดียวกันเสมอ** → ใช้วันที่เป็น `groups`

| วิธี | ข้อดี | ข้อเสีย |
|---|---|---|
| **`GroupShuffleSplit` (group = วันที่)** ✅ เลือก | สุ่มวันจากทั้งปี → test มีครบทุกฤดู | ไม่ได้จำลองการ "ทำนายอนาคต" |
| ตัดตามเวลา (ต้นปี = train, ท้ายปี = test) | สมจริงเชิงเวลา | ข้อมูลมีแค่ 1 ปี → test มีแค่ 1–2 ฤดูที่โมเดลไม่เคยเห็น |
| สุ่มทีละแถว ❌ | ง่าย | leakage |

เหตุผลที่เลือก: โจทย์ของเรา (ขั้น 1) คือ "อากาศ + เวลาแบบนี้ → ความต้องการสูงไหม" ไม่ใช่ forecasting จึงต้องการ test ที่ครอบคลุมทุกฤดู

### 3.1 แบ่งวันและบันทึกรายการวัน
**โค้ดทำอะไร:**
1. เรียงข้อมูลตาม `(date, hour)` ก่อน split → ลำดับแถวเหมือนกันทุกเครื่อง ผลการสุ่มจึงเหมือนกัน
2. `GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)` สุ่ม **20% ของจำนวนวัน** ไปเป็น test โดยส่ง `groups=df["date"]`
3. `assert` 2 ข้อ: ไม่มีวันไหนอยู่ทั้งสองฝั่ง และจำนวนวันรวมครบ 365
4. บันทึก **รายการวัน** (ไม่ใช่ตัวข้อมูล) เป็น ISO `yyyy-mm-dd` ลง `data/splits/` → notebook อื่นโหลดวันชุดเดียวกันได้ และ repo โครงงาน DS (`bike-trash-apps`) ใช้ logic เดียวกันเพื่อให้ได้วัน test ตรงกัน

In [7]:
from sklearn.model_selection import GroupShuffleSplit

df = df.sort_values(["date", "hour"]).reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=config.TEST_FRAC, random_state=config.RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df["date"]))

train_dates = np.sort(df.loc[train_idx, "date"].unique())
test_dates = np.sort(df.loc[test_idx, "date"].unique())
assert len(set(train_dates) & set(test_dates)) == 0, "มีวันที่อยู่ทั้ง train และ test"
assert len(train_dates) + len(test_dates) == df["date"].nunique()

for dates, path in [(train_dates, config.TRAIN_DATES), (test_dates, config.TEST_DATES)]:
    pd.Series(pd.to_datetime(dates).strftime("%Y-%m-%d"), name="date").to_csv(path, index=False)

n_days = df["date"].nunique()
print(f"train: {len(train_dates)} วัน ({len(train_dates) / n_days:.1%}) | {len(train_idx):,} แถว")
print(f"test : {len(test_dates)} วัน ({len(test_dates) / n_days:.1%}) | {len(test_idx):,} แถว")
print("วัน test 5 วันแรก:", list(pd.to_datetime(test_dates[:5]).strftime("%Y-%m-%d")))

train: 292 วัน (80.0%) | 7,008 แถว
test : 73 วัน (20.0%) | 1,752 แถว
วัน test 5 วันแรก: ['2017-12-01', '2017-12-04', '2017-12-06', '2017-12-10', '2017-12-16']


**ผลลัพธ์:**
- train **292 วัน (80.0%) = 7,008 แถว** · test **73 วัน (20.0%) = 1,752 แถว** — ทุกแถวของแต่ละวันไปอยู่ฝั่งเดียวกัน (assert ผ่าน)
- สัดส่วนแถวเท่ากับสัดส่วนวันพอดี เพราะทุกวันมี 24 แถว (ตรวจแล้วในขั้น 2.3)
- ไฟล์ `data/splits/train_dates.csv` และ `test_dates.csv` ถูกสร้าง → commit เข้า repo เป็น "สัญญา" ว่า test คือ 73 วันนี้ตลอดโปรเจกต์

### 3.2 ตรวจว่า split ใช้ได้
**โค้ดทำอะไร:**
- โหลดไฟล์รายการวันกลับมาแล้วเทียบกับค่าในหน่วยความจำ (ยืนยันว่าไฟล์ที่บันทึกถูกต้อง อ่านซ้ำได้)
- เทียบสัดส่วนฤดูกาลของ train vs test และนับวันพิเศษ (วันหยุด, วันที่ระบบปิด) ในแต่ละฝั่ง
- ⚠️ ดูแค่ **โครงสร้าง** ของ test (ฤดู/ประเภทวัน) — ไม่ดูยอดเช่าของ test เด็ดขาด

In [8]:
reloaded = pd.to_datetime(pd.read_csv(config.TEST_DATES)["date"]).values
assert (reloaded == test_dates).all()

side = np.where(df["date"].isin(test_dates), "test", "train")
season_share = pd.crosstab(df["seasons"], side, colnames=["side"], normalize="columns").mul(100).round(1)
print("สัดส่วนแถวตามฤดู (%)")
print(season_share, "\n")

day_info = df.assign(side=side).groupby("side").agg(
    days=("date", "nunique"),
    holiday_days=("date", lambda d: d[df.loc[d.index, "holiday"] == "Holiday"].nunique()),
    closed_hours=("functioning_day", lambda s: (s == "No").sum()),
)
day_info

สัดส่วนแถวตามฤดู (%)
side     test  train
seasons             
Autumn   27.4   24.3
Spring   19.2   26.7
Summer   21.9   26.0
Winter   31.5   22.9 



,days,holiday_days,closed_hours
side,,,
test,73,6,55
train,292,12,240


**ผลลัพธ์:**
- ไฟล์ที่บันทึกโหลดกลับมาตรงกับค่าเดิมทุกวัน
- **test มีครบทั้ง 4 ฤดู** (ซึ่งการตัดตามเวลาทำไม่ได้) แต่สัดส่วนไม่เท่า train: test มี Winter ~31.5% ขณะที่ train ~22.9% และ Spring น้อยกว่า (19.2% vs 26.7%)
  - นี่คือความแปรปรวนปกติของการสุ่มแค่ 73 วัน · **เราไม่แก้ split หลังเห็นตัวเลขนี้** (ถ้าสุ่มใหม่จนกว่าจะ "สวย" = การเลือก test ด้วยมือ)
  - ผลที่ตามมาที่ต้องจำไว้ตอนตีความ test: ฤดูหนาวยอดเช่าต่ำ → test น่าจะมีสัดส่วนชั่วโมง "ความต้องการสูง" น้อยกว่า train → precision/recall ของ test อาจต่างจาก CV ได้ และควรรายงานผล test แยกตามฤดูในขั้น error analysis
- วันหยุด: train 12 วัน, test 6 วัน · ชั่วโมงที่ระบบปิด: train 240, test 55

### 3.3 เก็บเฉพาะ train สำหรับ EDA
**โค้ดทำอะไร:** ตัดแถวของวัน train ออกมาเป็น `train` แล้ว **ลบ `df` (ข้อมูลทั้งชุดที่มี test ปนอยู่) ทิ้ง** กันเผลอใช้ในขั้น EDA

In [9]:
train = df[df["date"].isin(train_dates)].reset_index(drop=True)
del df, train_idx, test_idx
print("train:", train.shape, "| วัน:", train["date"].nunique())

train: (7008, 14) | วัน: 292


**ผลลัพธ์:** เหลือ `train` 7,008 แถว (292 วัน) — ทุกกราฟและตัวเลขในขั้น 4 มาจากตัวแปรนี้เท่านั้น